In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter as R

In [2]:
splitter = R(chunk_size=1000,chunk_overlap=200)

In [3]:
import pandas as pd
all_docs = pd.read_json(r'C:\Users\mehat\OneDrive - SRH\Subject\NLP\data\clean_data.json')

In [4]:
all_docs.shape

(56, 9)

In [5]:
chunks = []

for idx, row in all_docs.iterrows():

    text_chunks = splitter.split_text(
        row["clean_text"]
    )

    for i, chunk in enumerate(text_chunks):

        chunks.append({
            "chunk_id": f"{idx}_{i}",
            "text": chunk,
            "title": row["title"],
            "source": row["source"],
            "url": row["url"],
            "published": row["published"]
        })

In [6]:
chunks_df = pd.DataFrame(chunks)

print(chunks_df.shape)

(365, 6)


In [7]:
chunks_df["chunk_length"] = (
    chunks_df["text"]
    .str.len()
)

chunks_df["chunk_length"].describe()

count     365.000000
mean      937.810959
std       168.121538
min       201.000000
25%       992.000000
50%       996.000000
75%       998.000000
max      1000.000000
Name: chunk_length, dtype: float64

In [8]:
chunks_df.to_json(
    r"C:\Users\mehat\OneDrive - SRH\Subject\NLP\data\chunked_data.json",
    orient="records",
    indent=4
)

In [12]:
import numpy as np

np.save(
    r"C:\Users\mehat\OneDrive - SRH\Subject\NLP\data\chunk_embeddings.npy",
    embeddings
)

In [9]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
texts = chunks_df["text"].tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

In [15]:
embeddings.shape

(365, 384)

In [13]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(
    dimension
)

index.add(
    np.array(embeddings).astype("float32")
)

In [14]:
faiss.write_index(
    index,
    "sap_intelligence.index"
)

In [35]:
print(embeddings.shape)

(365, 384)


In [36]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(
    embeddings.astype("float32")
)

In [37]:
print(index.ntotal)

365


In [40]:
def retrieve(query, k=10):

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    scores, indices = index.search(
        query_embedding.astype("float32"),
        k
    )

    results = chunks_df.iloc[indices[0]]

    results = results.drop_duplicates(
        subset=["title"]
    )

    return results[
        ["title", "source", "text"]
    ]

In [41]:
retrieve(
    "What AI initiatives is SAP pursuing?"
)

,title,source,text
244,Agentic AI returns in India poised to grow fiv...,The Hitavada,"“In SAP, AI is grounded in deep industry conte..."
246,SAP to deploy 200 AI agents this year,The Hindu,"context and rich business semantics, said Mr.R..."
88,AI as a Game Changer for the Energy and Utilit...,SAP News,the opportunity to experience AI in real-world...
72,SAP’s AI-Native North Star Architecture: Techn...,SAP News,Bolting more intelligence onto isolated applic...
162,Operationalizing Autonomous CX with the Advanc...,SAP News,pace as new AI and platform capabilities becom...
229,Ericsson Scales AI Across the Enterprise with ...,SAP News,how co-innovation can create value beyond a si...
188,The Next Era of Business AI,SAP News,"employees, re‑engineering processes to connect..."


In [ ]:
def opportunity_agent(context):

    prompt = f"""
    You are a strategy analyst.

    Identify:

    1. Revenue opportunities
    2. Market opportunities
    3. Partnership opportunities
    4. AI opportunities

    Context:

    {context}
    """

    response = llm.invoke(prompt)

    return response.content